# Phase 2: Configuration Management Complete Guide

**Updated Measurement Configuration System for UppASD**

This comprehensive notebook demonstrates the new Phase 2 configuration system that simplifies parameter management from 200+ individual parameters to intuitive 1-3 line configurations.

## Quick Navigation

| Section | Purpose | Time |
|---------|---------|------|
| 1. Setup | Install dependencies | 2 min |
| 2. Helper Functions | Simplest approach (1-3 lines) | 5 min |
| 3. ConfigurationManager | Fine-grained control | 10 min |
| 4. ConfigurationBuilder | Fluent composition API | 10 min |
| 5. Validation & Exports | Error checking, formats | 5 min |
| 6. Real Examples | Complete working examples | 15 min |
| 7. Best Practices | Tips and patterns | 5 min |

**Total Time: ~50 minutes**

## Section 1: Setup & Imports

In [ ]:
# Import Phase 2 components
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Phase 2 imports
from uppasd import notebook
from uppasd.configuration_manager import ConfigurationManager, ConfigurationBuilder

print("✓ Phase 2 components imported successfully")
print(f"  - Helper functions: {dir(notebook)[:5]}...")
print(f"  - ConfigurationManager: Available")
print(f"  - ConfigurationBuilder: Available")

## Section 2: Helper Functions (Simplest - 1-3 Lines!)

In [ ]:
# Example 1: Basic measurement (energy + averages)
cfg_basic = notebook.measurement_basic()
print("Example 1: Basic measurement")
print(f"  Config: {cfg_basic}")
print(f"  Keys: {list(cfg_basic.keys())}")

# Example 2: Thermal properties (+ cumulants)
cfg_thermal = notebook.measurement_thermal()
print("\nExample 2: Thermal measurement")
print(f"  Config: {cfg_thermal}")

# Example 3: Spectroscopy (magnon spectra)
cfg_spec = notebook.measurement_spectroscopy()
print("\nExample 3: Spectroscopy")
print(f"  Config: {cfg_spec}")

### Helper Functions with Parameter Overrides

All 11 helper functions support parameter overrides via keyword arguments:

In [ ]:
# Override default parameters
cfg_custom = notebook.measurement_thermal(
    cumu_buff=20,  # Increase buffer size
    avrg_step=50   # More frequent averaging
)

print("Custom thermal configuration:")
print(f"  Cumulant buffer: {cfg_custom['measurement'].get('cumu_buff')}")
print(f"  Averaging step: {cfg_custom['measurement'].get('avrg_step')}")

# All 11 available helper functions
helpers = [
    'measurement_basic',
    'measurement_thermal',
    'measurement_spectroscopy',
    'measurement_correlations',
    'protocol_temperature_sweep',
    'protocol_3temperature',
    'protocol_spinice',
    'field_microwave',
    'interaction_stt',
    'analysis_wang_landau',
    'coupling_spinlattice'
]

print(f"\n✓ 11 helper functions available:")
for i, name in enumerate(helpers, 1):
    print(f"  {i:2d}. {name}()")

## Section 3: ConfigurationManager (Fine-Grained Control)

In [ ]:
# Initialize ConfigurationManager
cm = ConfigurationManager()

# Add measurement parameters
cm.add_measurement(
    plotenergy=2,
    do_avrg='Y',
    avrg_step=50
)

# Add spectroscopy
cm.add_spectroscopy(
    do_ams='Y',
    do_magdos='Y'
)

# Add temperature protocol
cm.add_protocol(
    do_tempexp='Y',
    tempexp_start=300.0,
    tempexp_end=50.0,
    tempexp_tau=10000.0
)

print("ConfigurationManager workflow:")
print(f"  Configuration: {cm}")
print(f"  Number of parameters: {sum(len(v) for v in cm.config.values() if isinstance(v, dict))}")
print(f"\nCategories: {list(cm.config.keys())}")

In [ ]:
# Method 2: Using presets
cm2 = ConfigurationManager()
cm2.add_preset('measurement_basic')
cm2.add_preset('field_microwave')

print("Using presets:")
print(f"  Added presets: measurement_basic + field_microwave")
print(f"  Result: {cm2.config}")

# Method 3: Direct dict initialization
config_dict = {
    'measurement': {'plotenergy': 1, 'do_avrg': 'Y'},
    'spectroscopy': {'do_ams': 'Y'}
}
cm3 = ConfigurationManager(config_dict)
print(f"\nDirect initialization:")
print(f"  Config: {cm3.config}")

## Section 4: ConfigurationBuilder (Fluent API)

In [ ]:
# ConfigurationBuilder - Method chaining for readability
cfg = (ConfigurationBuilder()
    .add_measurement('basic', avrg_step=100)
    .add_spectroscopy(do_ams='Y', do_magdos='Y')
    .add_protocol('temperature_sweep', tempexp_end=50.0)
    .add_field('microwave', mwf_freq=15.0)
    .build())

print("ConfigurationBuilder - Fluent API:")
print(f"  Configuration built: {cfg}")
print(f"  Type: {type(cfg).__name__}")
print(f"  Parameters: {sum(len(v) for v in cfg.config.values() if isinstance(v, dict))}")

# Access the underlying ConfigurationManager
print(f"\nUnderlying categories:")
for category, params in cfg.config.items():
    print(f"  {category}: {list(params.keys())}")

## Section 5: Validation & Export Formats

In [ ]:
# Validation example
cm_valid = ConfigurationManager()
cm_valid.add_measurement(plotenergy=1)
issues = cm_valid.validate()

print("✓ Valid configuration:")
print(f"  Issues: {issues if issues else 'None'}")

# Invalid configuration (conflicting protocols)
cm_invalid = ConfigurationManager()
cm_invalid.add_protocol(do_tempexp='Y', do_3tm='Y')  # Conflict!
issues = cm_invalid.validate()

print("\n✗ Invalid configuration (conflicting protocols):")
for issue in issues:
    print(f"  {issue}")

In [ ]:
# Export formats

# 1. Fortran-compatible kwargs (flat dict)
cm = ConfigurationManager().add_preset('measurement_basic')
kwargs = cm.to_fortran_kwargs()
print("1. Fortran kwargs (flat dict):")
print(f"   {kwargs}")
print()

# 2. JSON export
json_str = cm.to_json(indent=2)
print("2. JSON export:")
print(f"   {json_str[:100]}...")
print()

# 3. JSON import/export roundtrip
cm2 = ConfigurationManager.from_json(json_str)
print(f"3. JSON import - configs match: {cm.config == cm2.config}")
print()

# 4. YAML export (if PyYAML installed)
try:
    yaml_str = cm.to_yaml()
    print("4. YAML export:")
    print(f"   {yaml_str[:100]}...")
except ImportError:
    print("4. YAML export: (PyYAML not installed)")

## Section 6: Real-World Examples

### Example 1: Basic Spin Dynamics Simulation

In [ ]:
# Example 1: Simple energy and averages measurement
print("=" * 60)
print("EXAMPLE 1: Basic Spin Dynamics")
print("=" * 60)

cfg1 = notebook.measurement_basic()
kwargs1 = cfg1 if isinstance(cfg1, dict) else cfg1.config

print("\n✓ Configuration ready:")
print(f"  plotenergy: {kwargs1.get('plotenergy')}")
print(f"  do_avrg: {kwargs1.get('do_avrg')}")
print(f"  avrg_step: {kwargs1.get('avrg_step')}")

print("\n  Usage: sim.measure(**cfg)")
# Uncomment to run with sim:
# sim.measure(**cfg1)

### Example 2: Microwave-Driven Spectroscopy

In [ ]:
print("\n" + "=" * 60)
print("EXAMPLE 2: Microwave-Driven Spectroscopy")
print("=" * 60)

# Combine spectroscopy + microwave field
cfg2 = (ConfigurationBuilder()
    .add_measurement('spectroscopy')
    .add_field('microwave', mwf_freq=15.0, mwf_amp=0.05)
    .build())

print("\n✓ Configuration built with builder pattern:")
print(f"  Categories: {list(cfg2.config.keys())}")

kwargs2 = cfg2.to_fortran_kwargs()
print(f"\n  Fortran parameters: {len(kwargs2)} total")
print(f"  Sample params:")
print(f"    do_ams: {kwargs2.get('do_ams')}")
print(f"    mwf_freq: {kwargs2.get('mwf_freq')}")
print(f"    mwf_amp: {kwargs2.get('mwf_amp')}")

# Validate before running
issues = cfg2.validate()
print(f"\n  Validation: {'✓ PASS' if not issues else '✗ FAIL'}")
if issues:
    print(f"    Issues found: {issues}")

### Example 3: Temperature Sweep with Cooling Protocol

In [ ]:
print("\n" + "=" * 60)
print("EXAMPLE 3: Temperature Sweep + Cooling")
print("=" * 60)

# Helper function approach (simplest)
cfg3_helper = notebook.protocol_temperature_sweep(
    tempexp_start=500.0,
    tempexp_end=10.0,
    tempexp_tau=50000.0
)

# Manager approach (more control)
cm3 = ConfigurationManager()
cm3.add_preset('measurement_thermal')
cm3.add_protocol(
    do_tempexp='Y',
    tempexp_start=500.0,
    tempexp_end=10.0,
    tempexp_tau=50000.0,
    tempexp_step=1
)

print("\n✓ Configuration A (helper function):")
print(f"  {cfg3_helper}")

print("\n✓ Configuration B (ConfigurationManager):")
print(f"  Parameters: {sum(len(v) for v in cm3.config.values() if isinstance(v, dict))}")

# Both equivalent - choose what you prefer
print("\n  Both approaches produce compatible output for sim.measure()")

## Section 7: Best Practices & Recommendations

### Decision Tree: Which Approach to Use?

In [ ]:
print("""
DECISION GUIDE: Which API to Use?

┌─ Is this a STANDARD measurement pattern?
│  └─ YES → Use HELPER FUNCTIONS (simplest, 1-3 lines)
│     • measurement_basic()
│     • field_microwave()
│     • protocol_temperature_sweep()
│     
├─ Do you need CUSTOM COMBINATIONS?
│  └─ YES → Use CONFIGURATIONBUILDER (fluent API)
│     cfg = (ConfigurationBuilder()
│         .add_measurement('basic')
│         .add_field(mwf_freq=20.0)
│         .build())
│
└─ Do you need PROGRAMMATIC CONTROL & VALIDATION?
   └─ YES → Use CONFIGURATIONMANAGER (full features)
      cm = ConfigurationManager()
      cm.add_preset('measurement_basic')
      cm.add_field(custom_param=123)
      issues = cm.validate()
      kwargs = cm.to_fortran_kwargs()

RECOMMENDED FLOW:
1. Try helper function first → Is it sufficient?
2. If not, use ConfigurationBuilder → Is it readable?
3. If not, use ConfigurationManager → Full control!
""")

## Section 7: Best Practices & Decision Trees

### When to Use Each API

**Decision Tree:**

```
├─ Simple, standard scenario?
│  └─→ Use HELPER FUNCTIONS (1-3 lines)
│     - measurement_basic()
│     - field_microwave()
│     - Protocol overrides with kwargs
│
├─ Multiple presets or custom combinations?
│  └─→ Use CONFIGURATIONBUILDER (fluent API)
│     - Readable, chainable method calls
│     - Good for notebooks and visualization
│
└─ Fine-grained control or validation?
   └─→ Use CONFIGURATIONMANAGER (full control)
      - Manual parameter control
      - Schema validation
      - Multi-format export
```

### Best Practice 1: Always Validate Before Execution

```python
from uppasd.configuration_manager import ConfigurationManager

cm = ConfigurationManager()
cm.add_preset('measurement_spectroscopy')
cm.add_field(do_mwf='Y', mwf_freq=15.0)

# Always validate
issues = cm.validate()
if issues:
    print("Configuration issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("✓ Configuration is valid")
    cfg = cm.to_fortran_kwargs()
    sim.measure(**cfg)
```

### Best Practice 2: Override Parameters Explicitly

```python
# Better: Show exactly what's being changed
cfg = notebook.measurement_spectroscopy(
    do_magdos='Y',      # Explicitly override
    magdos_start=0.5,   # Make intent clear
    magdos_end=2.5
)

# Avoid: Hidden defaults
cfg = {'measurement': {'plotenergy': 1}}  # What about other params?
```

### Best Practice 3: Keep Configurations Modular

```python
# ✓ GOOD: Composable pieces
field_config = notebook.field_microwave(mwf_freq=15.0)
protocol_config = notebook.protocol_temperature_sweep(
    tempexp_start=1.0,
    tempexp_end=300.0
)

# Combine them
cm = ConfigurationManager()
cm.config.update(field_config)
cm.config.update(protocol_config)
```

### Best Practice 4: Document Your Configuration

```python
# Add comments explaining the physical meaning
config_comment = """
Configuration for microwave spectroscopy of Fe3O4 magnetic structure.

Parameters:
  - Microwave frequency: 15 GHz (X-band)
  - Spectroscopy mode: Magnetic moment dynamics
  - Averaging: 100 steps per measurement
  - Output: Averaged magnetic energy and structure factor
"""

cfg = notebook.field_microwave(mwf_freq=15.0)
print(config_comment)
```

### Best Practice 5: Version Your Configurations

```python
# Save to JSON with metadata
import json
from datetime import datetime

config_with_metadata = {
    'version': '2.0',
    'created': datetime.now().isoformat(),
    'description': 'Fe3O4 microwave spectroscopy',
    'configuration': notebook.field_microwave(mwf_freq=15.0)
}

with open('config_v2.0.json', 'w') as f:
    json.dump(config_with_metadata, f, indent=2)
```

## Section 8: Common Issues & Troubleshooting

### Issue 1: "Conflicting Protocols"

**Problem**: You tried to enable multiple temperature protocols

```python
# ✗ ERROR: Both enabled
cm.add_protocol(do_tempexp='Y', do_3tm='Y')
# Result: Validation error "Only one temperature protocol allowed"
```

**Solution**: Choose ONE protocol

```python
# ✓ CORRECT: Just one
cm.add_protocol(do_tempexp='Y')

# Or the 3-temperature model instead
cm.add_protocol(do_3tm='Y')
```

### Issue 2: "Structure Factor Requires Q-Vectors"

**Problem**: You specified do_sc='Q' but forgot to provide Q-points

```python
# ✗ ERROR
cm.add_measurement(do_sc='Q')
# Result: Validation error "Q-vectors required for structure factor"
```

**Solution**: Provide Q-vectors or use default lattice points

```python
# ✓ CORRECT: Provide Q-vectors
cm.add_measurement(
    do_sc='Q',
    qpoints=[[0, 0, 0], [0.5, 0.5, 0.5], [1, 1, 0]]
)

# OR: Use Q-file
cm.add_measurement(do_sc='Q', qfile='qpoints.dat')
```

### Issue 3: "Missing Parameter Types"

**Problem**: Wrong parameter type (string instead of float)

```python
# ✗ ERROR
cfg = notebook.measurement_basic(plotenergy='1')  # String!
# Result: Type mismatch

# ✓ CORRECT
cfg = notebook.measurement_basic(plotenergy=1)  # Integer
```

### Issue 4: "Cannot Chain After build()"

**Problem**: Trying to chain methods after build()

```python
# ✗ ERROR
cfg = ConfigurationBuilder().add_measurement('basic').build().add_field(...)
# Result: AttributeError - ConfigurationManager has no add_field method in chain

# ✓ CORRECT: Chain BEFORE build()
cfg = (ConfigurationBuilder()
    .add_measurement('basic')
    .add_field(do_mwf='Y')
    .build())
```

### Issue 5: "Parameters Not Showing in Fortran"

**Problem**: Fortran not reading your Phase 2 configuration

**Debug steps:**

```python
# Step 1: Check output dict
cfg = notebook.measurement_basic()
print("Configuration dict:", cfg)

# Step 2: Check flattened kwargs
cm = ConfigurationManager(cfg)
kwargs = cm.to_fortran_kwargs()
print("Fortran kwargs:", kwargs)

# Step 3: Verify sim.measure works with Phase 1
import uppasd
sim = uppasd.UppASD()
sim.measure(**kwargs)  # Should work from Phase 1
```

## Section 9: Summary & Next Steps

### What You've Learned

1. **Helper Functions** - Simplest approach for standard scenarios
   - 1-3 lines to get started
   - Pre-configured for 80% of use cases
   - 11 functions covering measurement, protocols, fields, analysis

2. **ConfigurationManager** - Full control and validation
   - Add parameters by category
   - Automatic schema validation
   - Export to Fortran kwargs, JSON, YAML
   - Load/save configurations

3. **ConfigurationBuilder** - Fluent API for readability
   - Chain method calls for clarity
   - Readable, notebook-friendly syntax
   - Still supports validation and export

4. **Validation** - Always check before execution
   - Schema validation catches type errors
   - Dependency checking ensures valid combinations
   - Custom validators can be added

5. **Integration with Phase 1** - Seamless with sim.measure()
   - Pass Phase 2 output to Phase 1 sim.measure()
   - All approaches output compatible dicts
   - Full validation pipeline

### Quick Reference

```python
# FASTEST: 1-line helper functions
cfg = notebook.measurement_basic()
sim.measure(**cfg)

# WITH OVERRIDE: Add parameters
cfg = notebook.field_microwave(mwf_freq=15.0)
sim.measure(**cfg)

# FLUENT API: Readable composition
cfg = (ConfigurationBuilder()
    .add_measurement('spectroscopy')
    .add_field('microwave', mwf_freq=15.0)
    .build())
sim.measure(**cfg)

# FULL CONTROL: Manual + validation
cm = ConfigurationManager()
cm.add_preset('measurement_thermal')
cm.add_field(do_mwf='Y')
issues = cm.validate()
if not issues:
    sim.measure(**cm.to_fortran_kwargs())
```

### Available Presets (Quick Reference)

| Category | Preset | Use Case |
|----------|--------|----------|
| Measurement | `measurement_basic` | Standard output |
| Measurement | `measurement_thermal` | Temperature dependence |
| Measurement | `measurement_spectroscopy` | Frequency domain |
| Measurement | `measurement_correlations` | Spatial correlations |
| Protocol | `protocol_temperature_sweep` | Tsweep protocol |
| Protocol | `protocol_3temperature` | 3-temp model |
| Protocol | `protocol_spinice` | Spin ice physics |
| Field | `field_microwave` | Microwave spectroscopy |
| Analysis | `analysis_wang_landau` | Wang-Landau sampling |
| Coupling | `coupling_stt` | Spin-transfer torque |
| Coupling | `coupling_spinlattice` | Spin-lattice coupling |

### Next Steps

1. **Modify for Your System**
   - Choose appropriate preset or helper function
   - Override parameters for your specific system
   - Test with validation

2. **Explore More Advanced Patterns**
   - Combine multiple presets with ConfigurationManager
   - Export configurations to JSON for tracking
   - Version your configurations

3. **Integrate with Your Workflows**
   - Use helper functions in quick scripts
   - Use ConfigurationBuilder in notebooks
   - Use ConfigurationManager for complex simulations

4. **Check the Documentation**
   - [API Reference](../PHASE2_API_REFERENCE.md) - Full documentation
   - [Troubleshooting Guide](../PHASE2_TROUBLESHOOTING.md) - FAQ and issues
   - [Quick Start](../PHASE2_QUICKSTART.md) - Getting started

### Resources

- **Phase 2 Documentation**: See /docs/Phase2/
- **Example Notebooks**: See /notebooks/
- **Unit Tests**: See /tests/test_configuration_manager.py (62 tests)
- **Source Code**: See /uppasd/configuration_manager.py

### Getting Help

If you encounter issues:

1. Check the [Troubleshooting Guide](../PHASE2_TROUBLESHOOTING.md)
2. Search the [FAQ](../PHASE2_TROUBLESHOOTING.md#frequently-asked-questions)
3. Run the example notebooks to see working code
4. Check test cases in test_configuration_manager.py
5. Read the API reference for detailed documentation

---

**Congratulations!** You now know how to use Phase 2 configuration management. 🎉